In [1]:
%env AWS_PROFILE=platform-developer

env: AWS_PROFILE=platform-developer


In [2]:
import json
import pickle
import re

import polars as pl

SIERRA_RAW_WORKS_PATH = "data/sierra-raw-works.parquet"
SIERRA_SOURCE_WORKS_PATH = "data/sierra-source-works.parquet"
FOLIO_RAW_BIBS_PATH = "data/folio-raw-bibs-29-09-2026.parquet"
FOLIO_SOURCE_WORKS_PATH = "data/folio-source-works-29-09-2026.parquet"

def from_parquet_snapshot(path: str):
    df = pl.read_parquet(path)
    return {row["id"]: pickle.loads(row["body"]) for row in df.iter_rows(named=True)}

In [3]:
sierra_raw_works = {
    row["id"]: json.loads(row["body"])
    for row in pl.read_parquet(SIERRA_RAW_WORKS_PATH).iter_rows(named=True)
}
print(f"Sierra raw works count: {len(sierra_raw_works)}")

Sierra raw works count: 1819104


In [23]:
folio_raw_bibs = dict(
    pl.read_parquet(FOLIO_RAW_BIBS_PATH, columns=["id", "content"]).iter_rows()
)
print(f"Folio raw bibs count: {len(folio_raw_bibs)}")

Folio raw bibs count: 1038657


In [3]:
sierra_source_works = from_parquet_snapshot(SIERRA_SOURCE_WORKS_PATH)
print(f"Sierra source works count: {len(sierra_source_works)}")

Sierra source works count: 1819104


In [4]:
folio_source_works = from_parquet_snapshot(FOLIO_SOURCE_WORKS_PATH)
print(f"Folio source works count: {len(folio_source_works)}")

Folio source works count: 1038462


In [7]:
from adapters.extractors.oai_pmh.folio import config as folio_config
from adapters.extractors.oai_pmh.folio.runtime import FOLIO_CONFIG

oai_client = FOLIO_CONFIG.build_oai_client()

def get_raw_folio_record(instance_id: str):
    return oai_client.get_record(
        instance_id, metadata_prefix=folio_config.OAI_METADATA_PREFIX
    )

In [5]:
folio_id_to_sierra_id = {}
for work_id, work in folio_source_works.items():
    if work.state.predecessor_identifier:
        sierra_id = work.state.predecessor_identifier.value
        folio_id_to_sierra_id[work_id] = sierra_id

# One Sierra bib resolves to one live Folio instance, so this reverse map is not
# lossy: the raw snapshot is a fresh OAI harvest, and instances superseded by an
# earlier Folio load are no longer in the repository. Build folio_source_works from
# the adapter store instead and it does become lossy - that table keeps every
# harvest generation, 2-3 rows per bib.
sierra_id_to_folio_id = {}
for k, v in folio_id_to_sierra_id.items():
    sierra_id_to_folio_id[v] = k

In [12]:
import unicodedata


def standardise_works_permanent(sierra_work, folio_work):
    """Normalise fields which we expect to differ between Folio and Sierra"""

    # Source identifiers will always be different
    del sierra_work["state"]["sourceIdentifier"]
    del folio_work["state"]["sourceIdentifier"]

    # Works no longer store siblings/children.
    # Popped rather than deleted: unlike CALM, most Sierra works are not hierarchical
    # and may carry no relations at all.
    relations = sierra_work["state"].get("relations", {})
    relations.pop("siblingsPreceding", None)
    relations.pop("siblingsSucceeding", None)
    relations.pop("children", None)

    # Source modified time will always be different
    del sierra_work["state"]["sourceModifiedTime"]
    del folio_work["state"]["sourceModifiedTime"]

    # Only Folio works have a predecessor identifier
    del folio_work["state"]["predecessorIdentifier"]

    # Only Folio works have a modified time
    del folio_work["state"]["modifiedTime"]

    # Versions will always be different
    del folio_work["version"]
    del sierra_work["version"]

    # Added automatically by the Elasticsearch ingest pipeline
    if "indexed_at" in sierra_work:
        del sierra_work["indexed_at"]

    # The Python pipeline uses the 'type' property more consistently than the Scala pipeline.
    for candidate in folio_work["state"].get("mergeCandidates", []):
        candidate["id"].pop("type", None)
    for subject in folio_work["data"].get("subjects", []):
        subject.pop("type", None)
    for item in folio_work["data"].get("production", []):
        for date in item.get("dates", []):
            date.pop("type", None)

    unicode_normalise_sierra_field(sierra_work, "title")
    unicode_normalise_sierra_field(sierra_work, "edition")
    unicode_normalise_sierra_field(sierra_work, "description")
    unicode_normalise_sierra_field(sierra_work, "alternativeTitles")
    unicode_normalise_sierra_field(sierra_work, "lettering")
    normalise_sierra_alternative_titles(sierra_work)
    normalise_sierra_physical_description(sierra_work)
    normalise_sierra_genres(sierra_work)
    normalise_sierra_production(sierra_work)
    normalise_subjects(sierra_work, folio_work)
    normalise_sierra_ancestors(sierra_work)
    normalise_sierra_contributors(sierra_work)

    return sierra_work, folio_work


def normalise_sierra_contributors(sierra_work):
    for c in sierra_work["data"]["contributors"]:
        c["agent"]["label"] = unicodedata.normalize("NFC", c["agent"]["label"])

        # source_id = c["agent"]["id"].get("sourceIdentifier")
        # if source_id and source_id["value"].startswith("n") and source_id["identifierType"]["id"] == "nlm-mesh":
        #     source_id["identifierType"]["id"] = "lc-names"

    

def unicode_normalise_sierra_field(sierra_work, field_name):
    val = sierra_work["data"].get(field_name)
    if isinstance(val, str):
        sierra_work["data"][field_name] = unicodedata.normalize("NFC", val)
    if isinstance(val, list):  
        sierra_work["data"][field_name] = [unicodedata.normalize("NFC", v) for v in val]


def normalise_subjects(sierra_work, folio_work):
    def _identity(entity):
        identifier = entity["id"].get("sourceIdentifier")
        if identifier is None:
            return ("label", entity["label"])
        return (
            identifier["identifierType"]["id"],
            identifier["ontologyType"],
            identifier["value"],
        )

    sierra_concepts = {
        (_identity(subject), _identity(concept)): concept
        for subject in sierra_work["data"]["subjects"]
        for concept in subject["concepts"]
    }

    # The Python parser intentionally handles more patterns and was validated independently.
    # If the FOLIO work has a parsed date range the Sierra work does not, remove the range.
    for subject in folio_work["data"]["subjects"]:
        for concept in subject["concepts"]:
            sierra_concept = sierra_concepts.get((_identity(subject), _identity(concept)))
            if sierra_concept is not None and "range" not in sierra_concept:
                concept.pop("range", None)

    # Unicode normalise Sierra labels
    for w in [sierra_work, folio_work]:
        for subject in w["data"]["subjects"]:
            for concept in subject["concepts"]:
                concept["label"] = unicodedata.normalize("NFC", concept["label"])
            subject["label"] = unicodedata.normalize("NFC", subject["label"])


def normalise_sierra_ancestors(sierra_work):
    for a in sierra_work["state"]["relations"]["ancestors"]:
        a["title"] = unicodedata.normalize("NFC", a["title"])

        
def normalise_sierra_physical_description(sierra_work):
    title = sierra_work["data"].get("physicalDescription")
    if title is not None:
        sierra_work["data"]["physicalDescription"] = sierra_work["data"]["physicalDescription"].rstrip()


def normalise_sierra_production(sierra_work):
    # Python always stores a 'type' property on concepts
    for production in sierra_work["data"]["production"]:
        production["label"] = unicodedata.normalize("NFC", production["label"]).rstrip()
        for place in production["places"]:
            place["type"] = "Place"
            place["label"] = unicodedata.normalize("NFC", place["label"]).strip()
        if production.get("function"):
            production["function"]["type"] = "Concept"
        for date in production["dates"]:
            if "range" in date:
                if date["range"]["from"] == "-9999-01-01T00:00:00Z":
                    date["range"]["from"] = "0001-01-01T00:00:00Z"
        for agent in production["agents"]:
            agent["label"] = unicodedata.normalize("NFC", agent["label"]).strip()


def normalise_sierra_alternative_titles(sierra_work):
    # Scala does not trim the joined subfields
    sierra_work["data"]["alternativeTitles"] = [
        strip_linkage_prefix(v).strip()
        for v in sierra_work["data"]["alternativeTitles"]
    ]

def normalise_sierra_genres(sierra_work):
    for genre in sierra_work["data"]["genres"]:
        for concept in genre["concepts"]:
            if concept["label"] == "Electronic Books":
                concept["label"] = "Electronic books"
    


LINKAGE_PREFIX = re.compile(r"^880-\d{2}(?:/[^\s]*)? ")                                                                                                                                                        


def strip_linkage_prefix(value: str) -> str:                                                                                                                                                                   
    """Remove the ǂ6 linkage number the Scala pipeline joins into the start of a field."""                                                                                                                              
    # return LINKAGE_PREFIX.sub("", value, count=1)
    return value
    

def standardise_works(sierra_work, folio_work):
    """Normalise some prevalent known issues/discrepancies to remove noise from the diff display."""

    sierra_work, folio_work = standardise_works_permanent(sierra_work, folio_work)    
    
    for w in (sierra_work, folio_work):
        w["data"].pop("title", None) # Confirmed matching. 39 differences are upstream.
        w["data"].pop("physicalDescription", None) # Confirmed matching. 48 differences are upstream.
        w["data"].pop("edition", None) # Confirmed matching. 2 differences are upstream.
        w["data"].pop("currentFrequency", None) # Confirmed matching. 2 differences due to Scala double space issue.
        w["data"].pop("formerFrequency", None) # Confirmed matching.
        w["data"].pop("designation", None) # Confirmed matching.
        w["data"].pop("description", None) # Confirmed matching. 20 differences are upstream.
        w["data"].pop("alternativeTitles", None) # Confirmed matching. 50 differences are upstream.
        w["data"].pop("duration", None) # TODO: Only 2 Folio works have this field. Must validate when more widely populated.
        w["data"].pop("genres", None) # Confirmed matching. 22 differences are upstream, 3 are intentional.
        w["data"].pop("production", None) # ~11k differences, all accounted for. Most are due to period parser differences.
        w["data"].pop("languages", None) # ~6k differences, most due to MARC 008 vs MARC 998 $f.
        w["data"].pop("subjects", None) # ~185 differences, 93 deliberate, the rest upstream. DATA QUALITY ISSUE: https://github.com/wellcomecollection/catalogue-pipeline/pull/3766
        w["state"].pop("relations", None) # Confirmed matching. ~400 differences upstream.
        w["data"].pop("lettering", None) # 1 whitespace difference. PICTURES DO NOT EXIST IN FOLIO.
        w["data"].pop("contributors", None) # 971 differences, 191 upstream, 781 deliberate.
        w["state"].pop("removedInternalWorkStubs", None) # Always empty in Python records, sometimes Nil in Sierra records due to outdated snapshot

        # w["data"].pop("collectionPath", None)
        w["data"].pop("notes", None)
        w["data"].pop("otherIdentifiers", None)
        
        w["state"].pop("mergeCandidates", None)
        w["data"].pop("items", None)
        w["data"].pop("holdings", None)

        # Blocked until MAT TYPE field equivalent becomes available
        w["data"].pop("format", None)
    
    return sierra_work, folio_work


# Folio works whose Sierra bib is absent from the snapshot. Expected: the Sierra
# snapshot is a point-in-time dump and some 907 b-numbers have no Sierra record at all.
missing_sierra_works = []


def stream_normalised_source_works(limit: int | None = None):
    for i, (work_id, work) in enumerate(folio_source_works.items()):
        if limit and i > limit:
            break

        if i % 10000 == 0 and i > 0:
            print(f"Streamed {i} works")

        if work.type == "Deleted" or not work.state.predecessor_identifier:
            continue

        if work.state.predecessor_identifier:
            sierra_id = work.state.predecessor_identifier.value
            sierra_work = sierra_source_works.get(sierra_id)

            if sierra_work is None:
                missing_sierra_works.append(work_id)
                continue

            if sierra_work["type"] in {"Deleted", "Invisible"}:
                continue

            sierra_work, folio_work = standardise_works(copy.deepcopy(sierra_work), work.model_dump(exclude_none=True))
            yield {"id": sierra_id, "work": sierra_work}, {"id": work_id, "work": folio_work}

In [14]:
from deepdiff import DeepDiff
import re
import copy
from collections import Counter

matching_works = 0
diff_works = 0

range_labels = Counter()

for sierra, folio in stream_normalised_source_works():
    sierra_id, sierra_work = sierra["id"], sierra["work"]
    folio_id, folio_work = folio["id"], folio["work"]

    diffs = DeepDiff(sierra_work, folio_work, ignore_order=True)
    if diffs:
        print(folio_id, sierra_id)
        diff_works += 1

        if "values_changed" in diffs:
            for key, val in diffs["values_changed"].items():
                print(key)
                print(f'\t {val["old_value"]}')
                print(f'\t {val["new_value"]}')
        if "iterable_item_removed" in diffs:
            print("ITEM REMOVED")
            print(diffs["iterable_item_removed"])
        if "iterable_item_added" in diffs:
            print("ITEM ADDED")
            print(diffs["iterable_item_added"])
        if "dictionary_item_removed" in diffs:
            print("DICTIONARY ITEM REMOVED")
            print(diffs["dictionary_item_removed"])
        if "dictionary_item_added" in diffs:
            print("DICTIONARY ITEM ADDED")
            print(diffs["dictionary_item_added"])
            

        print("------------------------------------")
    else:
        matching_works += 1

print(f"Matching works: {matching_works}")
print(f"Diff works: {diff_works}")
print(f"Folio works whose Sierra bib is missing from the snapshot: {len(missing_sierra_works)}")

oai:edge-wellcome.folio.ebsco.com:fs00001190/0004b556-6edf-5587-91c7-3e364d4a714d b13587018
DICTIONARY ITEM REMOVED
SetOrdered(["root['data']['collectionPath']"])
------------------------------------
oai:edge-wellcome.folio.ebsco.com:fs00001190/000f9dfe-5e8f-570d-be04-7b7ab3bba054 b13580863
DICTIONARY ITEM REMOVED
SetOrdered(["root['data']['collectionPath']"])
------------------------------------
oai:edge-wellcome.folio.ebsco.com:fs00001190/00109661-551a-5b44-a448-0547c9ea00f0 b13865067
DICTIONARY ITEM REMOVED
SetOrdered(["root['data']['collectionPath']"])
------------------------------------
oai:edge-wellcome.folio.ebsco.com:fs00001190/0023b109-76a0-5b0a-aec4-3b468c894901 b13869462
DICTIONARY ITEM REMOVED
SetOrdered(["root['data']['collectionPath']"])
------------------------------------
oai:edge-wellcome.folio.ebsco.com:fs00001190/0025d9dc-e4c5-5fa9-a9ad-3e3960f2cbb5 b13628719
DICTIONARY ITEM REMOVED
SetOrdered(["root['data']['collectionPath']"])
------------------------------------


In [ ]:
deleted_sierra_works = []
deleted_sierra_works_without_folio_match = []
sierra_works_without_folio_match = []
deleted_folio_works = []
no_predecessor_id = []
sierra_id_not_in_snapshot = []

for work_id, work in folio_source_works.items():
    if work.type == "Deleted":
        deleted_folio_works.append(work_id)
        continue

    if work.state.predecessor_identifier:
        sierra_id = work.state.predecessor_identifier.value
        sierra_work = sierra_source_works.get(sierra_id)

        if sierra_work is None:
            sierra_id_not_in_snapshot.append(work_id)
            continue

        if sierra_work["type"] == "Deleted":
            deleted_sierra_works.append(work_id)
            continue
    else:
        no_predecessor_id.append(work_id)

for work_id, work in sierra_source_works.items():
    if work_id in sierra_id_to_folio_id:
        continue

    if work["type"] == "Deleted":
        deleted_sierra_works_without_folio_match.append(work_id)
    else:
        sierra_works_without_folio_match.append(work_id)

print(f"Works deleted in Folio: {len(deleted_folio_works)}")
print(f"Works deleted in Sierra but visible in Folio: {len(deleted_sierra_works)}")
print(f"Folio works without a predecessor identifier: {len(no_predecessor_id)}")
print(f"Folio works whose Sierra bib is missing from the snapshot: {len(sierra_id_not_in_snapshot)}")

print(f"Deleted Sierra works without a matching Folio work: {len(deleted_sierra_works_without_folio_match)}")
print(f"Non-deleted Sierra works without a matching Folio work: {len(sierra_works_without_folio_match)}")